<a href="https://colab.research.google.com/github/dakshini01/Statistical-Learning-e20181/blob/main/Assignment_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Maximum Entropy Inference Assignment — Complete Solutions

This notebook gives complete derivations and executable verification for all four questions.

## Consistency note for Question 3

The assignment says “four destinations,” but only two classes are defined:

- \(Y=0\): Finance
- \(Y=1\): Technical

So the problem is solved as a **binary maximum-entropy classifier**.

The five listed state probabilities sum to \(0.90\). The remaining \(0.10\) is treated as an implied no-context state

\[
x_F=(0,0,0)^T,\qquad P(x_F)=0.10.
\]

This does not alter the three supplied feature constraints and is consistent with the statement that the total mass for \(x_3=1\) is \(0.20\).


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.optimize import root

np.set_printoptions(precision=10, suppress=True)



# Question 1 — Quantum Harmonic Oscillator

The energy levels are

\[
E_j=\hbar\omega\left(j+\frac12\right),\qquad j=0,1,2,\ldots
\]

Let

\[
\beta=\frac1{k_BT},
\qquad
x=\beta\hbar\omega=\frac{\hbar\omega}{k_BT}.
\]

## 1. Partition function

\[
\begin{aligned}
Z(T)
&=\sum_{j=0}^{\infty}e^{-\beta E_j}\\
&=\sum_{j=0}^{\infty}
e^{-\beta\hbar\omega(j+1/2)}\\
&=e^{-\beta\hbar\omega/2}
\sum_{j=0}^{\infty}
\left(e^{-\beta\hbar\omega}\right)^j.
\end{aligned}
\]

This is a geometric series with ratio \(r=e^{-\beta\hbar\omega}\). Since \(T>0\), \(0<r<1\), so

\[
\boxed{
Z(T)=
\frac{e^{-\hbar\omega/(2k_BT)}}
{1-e^{-\hbar\omega/(k_BT)}}.
}
\]

Equivalently,

\[
Z(T)=
\frac1{2\sinh\left(\frac{\hbar\omega}{2k_BT}\right)}.
\]

## 2. Helmholtz free energy and internal energy

Since

\[
\ln Z
=
-\frac{\hbar\omega}{2k_BT}
-\ln\left(1-e^{-\hbar\omega/(k_BT)}\right),
\]

\[
\boxed{
F(T)
=
-k_BT\ln Z
=
\frac{\hbar\omega}{2}
+
k_BT\ln\left(1-e^{-\hbar\omega/(k_BT)}\right).
}
\]

Also,

\[
U=-\frac{\partial\ln Z}{\partial\beta},
\]

so

\[
\boxed{
U(T)
=
\frac{\hbar\omega}{2}
+
\frac{\hbar\omega}
{e^{\hbar\omega/(k_BT)}-1}.
}
\]

Equivalent form:

\[
U(T)=
\frac{\hbar\omega}{2}
\coth\left(\frac{\hbar\omega}{2k_BT}\right).
\]

## 3. Entropy

Using \(S=-(\partial F/\partial T)\),

\[
\boxed{
S(T)
=
k_B\left[
\frac{x}{e^x-1}
-\ln(1-e^{-x})
\right],
\qquad
x=\frac{\hbar\omega}{k_BT}.
}
\]

Verification:

\[
\begin{aligned}
F+TS
&=
\frac{\hbar\omega}{2}
+k_BT\ln(1-e^{-x})\\
&\quad+
k_BT\left[
\frac{x}{e^x-1}
-\ln(1-e^{-x})
\right]\\
&=
\frac{\hbar\omega}{2}
+
\frac{k_BTx}{e^x-1}\\
&=
\frac{\hbar\omega}{2}
+
\frac{\hbar\omega}{e^x-1}
=
U.
\end{aligned}
\]

Therefore,

\[
\boxed{U=F+TS.}
\]

## 4. Heat capacity and hyperbolic form

\[
C_V=\left(\frac{\partial U}{\partial T}\right)_V
\]

gives

\[
\boxed{
C_V
=
k_Bx^2\frac{e^x}{(e^x-1)^2}.
}
\]

Since

\[
e^x-1=2e^{x/2}\sinh(x/2),
\]

\[
(e^x-1)^2=4e^x\sinh^2(x/2).
\]

Therefore,

\[
\boxed{
C_V
=
k_B\left(\frac{x}{2}\right)^2
\frac1{\sinh^2(x/2)}
=
k_B
\left(\frac{\hbar\omega}{2k_BT}\right)^2
\frac1{
\sinh^2\left(\frac{\hbar\omega}{2k_BT}\right)
}.
}
\]

## 5. High-temperature limit

When \(k_BT\gg\hbar\omega\), \(x\ll1\), and

\[
\sinh(x/2)\sim x/2.
\]

Thus

\[
C_V\sim
k_B\left(\frac{x}{2}\right)^2
\frac1{(x/2)^2}
=
k_B.
\]

Hence

\[
\boxed{\lim_{T\to\infty}C_V=k_B.}
\]

This matches the classical equipartition result. A one-dimensional harmonic oscillator has one quadratic kinetic term and one quadratic potential term, so \(U_{\text{classical}}=k_BT\) and \(C_V=k_B\).


In [ ]:

# Q1 numerical verification in natural units: hbar = omega = k_B = 1

def oscillator_quantities(T, hbar=1.0, omega=1.0, kB=1.0):
    x = hbar * omega / (kB * T)
    Z = np.exp(-x / 2) / (1 - np.exp(-x))
    F = 0.5 * hbar * omega + kB * T * np.log(1 - np.exp(-x))
    U = 0.5 * hbar * omega + hbar * omega / np.expm1(x)
    S = kB * (x / np.expm1(x) - np.log(1 - np.exp(-x)))
    Cv = kB * x**2 * np.exp(x) / np.expm1(x)**2
    Cv_hyperbolic = kB * (x / 2)**2 / np.sinh(x / 2)**2
    return Z, F, U, S, Cv, Cv_hyperbolic

rows = []
for T in [0.2, 1.0, 10.0, 100.0]:
    Z, F, U, S, Cv, Cv_h = oscillator_quantities(T)
    rows.append({
        "T": T,
        "Z": Z,
        "F": F,
        "U": U,
        "F + T*S": F + T*S,
        "Cv": Cv,
        "Cv hyperbolic": Cv_h,
    })

display(pd.DataFrame(rows))



# Question 2 — Maximum Entropy with Fixed Mean and Variance

We maximize differential entropy

\[
h[p]=-\int_{-\infty}^{\infty}p(x)\ln p(x)\,dx
\]

subject to

\[
\int p(x)\,dx=1,
\qquad
\int xp(x)\,dx=\mu,
\qquad
\int (x-\mu)^2p(x)\,dx=\sigma^2.
\]

The functional Lagrangian is

\[
\begin{aligned}
\mathcal L[p]
&=
-\int p(x)\ln p(x)\,dx\\
&\quad
-\lambda_0\left(\int p(x)\,dx-1\right)\\
&\quad
-\lambda_1\left(\int xp(x)\,dx-\mu\right)\\
&\quad
-\lambda_2\left(\int (x-\mu)^2p(x)\,dx-\sigma^2\right).
\end{aligned}
\]

The functional derivative is

\[
\frac{\delta\mathcal L}{\delta p(x)}
=
-\ln p(x)-1-\lambda_0-\lambda_1x-\lambda_2(x-\mu)^2.
\]

Setting it equal to zero gives

\[
p^*(x)
=
\frac1Z
\exp\left[-\lambda_1x-\lambda_2(x-\mu)^2\right].
\]

Because the required mean is \(\mu\), the centered form has

\[
\lambda_1=0.
\]

The variance constraint gives

\[
\lambda_2=\frac1{2\sigma^2}.
\]

The normalization constant is

\[
Z=
\int_{-\infty}^{\infty}
\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right]dx
=
\sqrt{2\pi\sigma^2}.
\]

Therefore,

\[
\boxed{
p^*(x)
=
\frac1{\sqrt{2\pi\sigma^2}}
\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right].
}
\]

Hence

\[
\boxed{X\sim\mathcal N(\mu,\sigma^2).}
\]

The entropy functional is strictly concave and the constraints are linear, so this stationary solution is the unique global maximum.


In [ ]:

# Q2 numerical verification

mu = 2.0
sigma = 1.5

def gaussian_pdf(x):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (
        np.sqrt(2 * np.pi) * sigma
    )

normalization = quad(gaussian_pdf, -np.inf, np.inf)[0]
mean_check = quad(lambda x: x * gaussian_pdf(x), -np.inf, np.inf)[0]
variance_check = quad(
    lambda x: (x - mu)**2 * gaussian_pdf(x),
    -np.inf,
    np.inf,
)[0]

print("Normalization =", normalization)
print("Mean          =", mean_check)
print("Variance      =", variance_check)
print("Entropy       =", 0.5 * np.log(2 * np.pi * np.e * sigma**2))



# Question 3 — Maximum-Entropy Email Classification

Let

\[
q(x)=P(Y=1\mid x),
\]

where \(Y=1\) means Technical.

We maximize

\[
H(Y\mid X)
=
-\sum_xP(x)
\left[
q(x)\ln q(x)
+
(1-q(x))\ln(1-q(x))
\right]
\]

subject to

\[
\sum_xP(x)x_1q(x)=0.05,
\]

\[
\sum_xP(x)x_2q(x)=0.15,
\]

\[
\sum_xP(x)x_3q(x)=0.16.
\]

Introducing multipliers \(w=(w_1,w_2,w_3)^T\), differentiation with respect to \(q(x)\) gives

\[
\ln\left(\frac{1-q(x)}{q(x)}\right)+w^Tx=0.
\]

Therefore,

\[
\ln\left(\frac{q(x)}{1-q(x)}\right)=w^Tx
\]

and

\[
\boxed{
P(Y=1\mid x)
=
q(x)
=
\frac1{1+\exp(-w^Tx)}.
}
\]

For the five supplied states,

\[
\begin{aligned}
q_A&=\sigma(w_1),\\
q_B&=\sigma(w_2),\\
q_C&=\sigma(w_3),\\
q_D&=\sigma(w_2+w_3),\\
q_E&=\sigma(w_1+w_2).
\end{aligned}
\]

The implied no-context state has

\[
q_F=\sigma(0)=0.5.
\]

The moment equations are

\[
0.40q_A+0.05q_E=0.05,
\]

\[
0.25q_B+0.15q_D+0.05q_E=0.15,
\]

\[
0.05q_C+0.15q_D=0.16.
\]

Numerical solution:

\[
\boxed{
w_1=-1.97246747,\quad
w_2=-1.77620568,\quad
w_3=2.88039264.
}
\]

Thus

\[
\boxed{
P(Y=1\mid x)
=
\sigma(
-1.97246747x_1
-1.77620568x_2
+2.88039264x_3
).
}
\]


In [ ]:

# Q3 numerical solution and verification

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

states = np.array([
    [1, 0, 0],  # A
    [0, 1, 0],  # B
    [0, 0, 1],  # C
    [0, 1, 1],  # D
    [1, 1, 0],  # E
    [0, 0, 0],  # F: implied residual state
], dtype=float)

prob_x = np.array([0.40, 0.25, 0.05, 0.15, 0.05, 0.10])

names = [
    "A: Pure Financial",
    "B: Pure Access",
    "C: Pure Technical",
    "D: Access + Technical",
    "E: Billing + Access",
    "F: No detected context",
]

targets = np.array([0.05, 0.15, 0.16])

def equations(w):
    q = sigmoid(states @ w)
    fitted = np.array([
        np.sum(prob_x * states[:, 0] * q),
        np.sum(prob_x * states[:, 1] * q),
        np.sum(prob_x * states[:, 2] * q),
    ])
    return fitted - targets

solution = root(equations, x0=np.array([-2.0, -1.5, 3.0]))

if not solution.success:
    raise RuntimeError(solution.message)

w = solution.x
q_tech = sigmoid(states @ w)

result_table = pd.DataFrame({
    "Scenario": names,
    "P(x)": prob_x,
    "P(Technical | x)": q_tech,
    "P(Finance | x)": 1 - q_tech,
})

print("Weights:", w)
print("Constraint residuals:", equations(w))
display(result_table)

print("P(Technical) =", np.sum(prob_x * q_tech))
print("P(Finance)   =", 1 - np.sum(prob_x * q_tech))



The scenario-level Technical probabilities are approximately:

| Scenario | \(P(\text{Technical}\mid x)\) |
|---|---:|
| Pure Financial | \(0.1221\) |
| Pure Access | \(0.1448\) |
| Pure Technical | \(0.9469\) |
| Access + Technical | \(0.7510\) |
| Billing + Access | \(0.0230\) |
| No detected context | \(0.5000\) |

The strong positive value of \(w_3\) shows that bug or malfunction evidence strongly increases Technical routing. The fitted model reproduces all three supplied expectations exactly.



# Question 4 — Maximum-Entropy Asset-Price Density

We maximize

\[
h[p]
=
-\int_0^\infty p(s)\ln p(s)\,ds
\]

subject to

\[
\int_0^\infty p(s)\,ds=1,
\]

\[
E[S_T]=102,
\qquad
E[S_T^2]=10600,
\qquad
E[\ln S_T]=4.60.
\]

The functional Lagrangian is

\[
\begin{aligned}
\mathcal L[p]
&=
-\int_0^\infty p(s)\ln p(s)\,ds\\
&\quad
-\gamma\left(\int_0^\infty p(s)\,ds-1\right)\\
&\quad
-\lambda_1\left(\int_0^\infty sp(s)\,ds-102\right)\\
&\quad
-\lambda_2\left(\int_0^\infty s^2p(s)\,ds-10600\right)\\
&\quad
-\lambda_3\left(\int_0^\infty\ln(s)p(s)\,ds-4.60\right).
\end{aligned}
\]

The functional derivative is

\[
\frac{\delta\mathcal L}{\delta p(s)}
=
-\ln p(s)-1-\gamma
-\lambda_1s-\lambda_2s^2-\lambda_3\ln s.
\]

Therefore,

\[
\boxed{
p^*(s)
=
\frac1Z
s^{-\lambda_3}
\exp(-\lambda_1s-\lambda_2s^2),
\qquad s>0.
}
\]

For integrability,

\[
\lambda_2>0,
\qquad
\lambda_3<1.
\]

## Stable rescaling

Let

\[
x=\frac{s}{100}.
\]

Then

\[
E[X]=1.02,
\qquad
E[X^2]=1.06,
\qquad
E[\ln X]=4.60-\ln100.
\]

Write

\[
q(x)
=
\frac1{Z_x}
x^{a-1}e^{-bx-cx^2},
\qquad x>0,
\]

where

\[
a=1-\lambda_3,
\qquad
b=100\lambda_1,
\qquad
c=10000\lambda_2.
\]

Define

\[
Z(a,b,c)
=
\int_0^\infty x^{a-1}e^{-bx-cx^2}\,dx.
\]

Using the parabolic-cylinder function \(D_\nu\),

\[
Z(a,b,c)
=
(2c)^{-a/2}
\Gamma(a)
e^{b^2/(8c)}
D_{-a}\left(\frac{b}{\sqrt{2c}}\right).
\]

The moments are

\[
E[X^r]
=
\frac{Z(a+r,b,c)}{Z(a,b,c)}
\]

and

\[
E[\ln X]
=
\frac{\partial}{\partial a}\ln Z(a,b,c).
\]

Solving the three equations gives

\[
\boxed{
a=0.00001095553815,
}
\]

\[
\boxed{
b=-54.0842118540,
}
\]

\[
\boxed{
c=26.0216542673.
}
\]

Therefore,

\[
\lambda_1=-0.540842118540,
\]

\[
\lambda_2=0.00260216542673,
\]

\[
\lambda_3=0.999989044462.
\]

The density on the original dollar scale is

\[
\boxed{
p(s)
=
\frac1{Z_s}
s^{-0.999989044462}
\exp\left(
0.540842118540s
-
0.00260216542673s^2
\right),
\qquad s>0,
}
\]

where

\[
\boxed{
Z_s\approx5.45912014069\times10^{11}.
}
\]

This density satisfies

\[
E[S_T]=102,
\qquad
E[S_T^2]=10600,
\qquad
E[\ln S_T]=4.60.
\]

The logarithmic constraint pushes \(\lambda_3\) very close to the integrability boundary \(1\). The result has a main peak near the forward-price region and a very small near-zero crash component, representing the imposed downside skew.


In [ ]:

# Q4 numerical solution using arbitrary precision

import mpmath as mp

mp.mp.dps = 60

target_mean_x = mp.mpf("1.02")
target_second_x = mp.mpf("1.06")
target_log_x = mp.mpf("4.60") - mp.log(100)

def log_Z_scaled(a, b, c):
    # Z(a,b,c) = integral x^(a-1) exp(-b*x-c*x^2) dx, x>0
    z = b / mp.sqrt(2 * c)
    return (
        -(a / 2) * mp.log(2 * c)
        + mp.loggamma(a)
        + b**2 / (8 * c)
        + mp.log(mp.pcfd(-a, z))
    )

def scaled_moments(a, b, c):
    log_z0 = log_Z_scaled(a, b, c)

    mean_x = mp.e ** (
        log_Z_scaled(a + 1, b, c) - log_z0
    )

    second_x = mp.e ** (
        log_Z_scaled(a + 2, b, c) - log_z0
    )

    mean_log_x = mp.diff(
        lambda aa: log_Z_scaled(aa, b, c),
        a,
    )

    return mean_x, second_x, mean_log_x

def f1(a, b, c):
    return scaled_moments(a, b, c)[0] - target_mean_x

def f2(a, b, c):
    return scaled_moments(a, b, c)[1] - target_second_x

def f3(a, b, c):
    return scaled_moments(a, b, c)[2] - target_log_x

a_sol, b_sol, c_sol = mp.findroot(
    (f1, f2, f3),
    (
        mp.mpf("2e-5"),
        mp.mpf("-53"),
        mp.mpf("25.5"),
    ),
    tol=mp.mpf("1e-35"),
    maxsteps=100,
)

lambda_1 = b_sol / 100
lambda_2 = c_sol / 10000
lambda_3 = 1 - a_sol

log_Z_x = log_Z_scaled(a_sol, b_sol, c_sol)
log_Z_s = a_sol * mp.log(100) + log_Z_x
Z_s = mp.e ** log_Z_s

mean_x, second_x, mean_log_x = scaled_moments(
    a_sol, b_sol, c_sol
)

print("a =", mp.nstr(a_sol, 20))
print("b =", mp.nstr(b_sol, 20))
print("c =", mp.nstr(c_sol, 20))

print("\nlambda_1 =", mp.nstr(lambda_1, 20))
print("lambda_2 =", mp.nstr(lambda_2, 20))
print("lambda_3 =", mp.nstr(lambda_3, 20))
print("Z_s      =", mp.nstr(Z_s, 20))

print("\nConstraint verification")
print("E[S]     =", mp.nstr(100 * mean_x, 20))
print("E[S^2]   =", mp.nstr(10000 * second_x, 20))
print("E[ln S]  =", mp.nstr(mean_log_x + mp.log(100), 20))
print("Var(S)   =", mp.nstr(
    10000 * second_x - (100 * mean_x)**2,
    20,
))


In [ ]:

# Q4 density plot in the main price region

a_float = float(a_sol)
lambda_1_float = float(lambda_1)
lambda_2_float = float(lambda_2)
log_Z_s_float = float(log_Z_s)

def log_asset_density(s):
    s = np.asarray(s, dtype=float)
    return (
        (a_float - 1) * np.log(s)
        - lambda_1_float * s
        - lambda_2_float * s**2
        - log_Z_s_float
    )

s = np.linspace(0.25, 170, 2500)
density = np.exp(log_asset_density(s))

plt.figure(figsize=(8, 5))
plt.plot(s, density)
plt.xlabel(r"Future asset price $S_T$")
plt.ylabel(r"Density $p(S_T)$")
plt.title("Maximum-Entropy Asset-Price Density")
plt.grid(True, alpha=0.3)
plt.show()
